In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [2]:
df = pd.read_excel('../data/Online Retail.xlsx')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


- Quantity : 고갯이 한번에 주문한 상품의 수량
- InvoiceDate : 주문 날짜
- UnitPrice : 상품 1개당 가격

In [4]:
# 고객별 빈도수를 확인하기 위해서 고객ID가 존재하지 않는 결측치는 제거
df.dropna(subset = 'CustomerID', inplace = True)

In [5]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,406829.000000,406829,406829.000000,406829.000000
mean,12.061303,2011-07-10 16:30:57.879207424,3.460471,15287.690570
min,-80995.000000,2010-12-01 08:26:00,0.000000,12346.000000
25%,2.000000,2011-04-06 15:02:00,1.250000,13953.000000
50%,5.000000,2011-07-31 11:48:00,1.950000,15152.000000
75%,12.000000,2011-10-20 13:06:00,3.750000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,248.693370,NaN,69.315162,1713.600303


In [6]:
df = df.loc[
    df['Quantity'] > 0
]

In [7]:
df.loc[df['Quantity'] > 10000]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom


In [8]:
# 최근성, 빈도, 총지출
# 최근성 -> 주문일자 + 1일 (현재 시간)
#           현재시간 - 주문일자
# 빈도 -> 고객ID당 거래 횟수
# 총지출 -> 물건의 개수 * 물건 1개당 가격 -> id별 지출의 합계

# 거래 금액(price) 생성
df['price'] = df['Quantity'] * df['UnitPrice']

In [9]:
# 현재 시간을 생성
now_time = df['InvoiceDate'].max() + pd.Timedelta(days=1)
now_time

Timestamp('2011-12-10 12:50:00')

In [10]:
now_time - df['InvoiceDate']

0        374 days 04:24:00
1        374 days 04:24:00
2        374 days 04:24:00
3        374 days 04:24:00
4        374 days 04:24:00
                ...       
541904     1 days 00:00:00
541905     1 days 00:00:00
541906     1 days 00:00:00
541907     1 days 00:00:00
541908     1 days 00:00:00
Name: InvoiceDate, Length: 397924, dtype: timedelta64[ns]

In [11]:
customers = df.groupby('CustomerID').agg(
    {
        'InvoiceDate' : lambda x : (now_time - x.max()).days,
        'InvoiceNo' : 'count',
        'price' : 'sum'
    }
)
customers

,InvoiceDate,InvoiceNo,price
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40
...,...,...,...
18280.0,278,10,180.60
18281.0,181,7,80.82
18282.0,8,12,178.05


In [12]:
customers.columns = ['최근성', '빈도', '총지출']

In [13]:
x = customers[['최근성', '빈도']]
y = customers['총지출']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

In [15]:
model = RandomForestRegressor(random_state = 42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
pred = model.predict(X_test)

In [17]:
from sklearn.metrics import r2_score

In [18]:
print(r2_score(y_test, pred))

-0.4886041700567185


In [19]:
importance_df = pd.DataFrame(
    {
        'feature_name' : x.columns,
        'importance' : model.feature_importances_
    }
).sort_values('importance', ascending = False)

In [20]:
importance_df

,feature_name,importance
1,빈도,0.816349
0,최근성,0.183651
